# 02 - Data Extraction

**Purpose:** Extract structured data from scraped HTML using regex patterns.

| | |
|---|---|
| **Inputs** | `data/pilot/*.html` (500 HTML files) |
| **Outputs** | `data/pilot/extracted.json`, `data/pilot/validation_results.json` |
| **Runtime** | ~30 seconds |

## Pipeline stages
1. Structured fields from metadata table (court, judge, date, crime article)
2. Demographics from bio section (gender, age, education, occupation)
3. Sentencing from decision section (type, duration, fine amount)
4. Validation against independent extraction

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import json
import re

from extractor import (
    extract_case,
    extract_structured_fields,
    extract_bio_section,
    extract_sentence_section,
    extract_demographics_regex,
    extract_sentence_regex,
    process_batch,
    parse_html,
    CaseData,
)

## 1. Examine HTML Structure

Each case page has:
- A **metadata table** with court, judge, case number, date, crime article
- A **bio section** ("биеийн байцаалт") with defendant demographics
- A **decision section** ("ТОГТООХ") with the sentence

In [ ]:
# Load a sample case
sample_path = Path('../data/pilot/100480.html')
html = sample_path.read_text(encoding='utf-8')
soup = parse_html(html)
text = soup.get_text()

print(f"Case ID: {sample_path.stem}")
print(f"Text length: {len(text):,} chars")
print()

# Show the metadata table
table = soup.find('table')
if table:
    print("=== Metadata Table ===")
    for row in table.find_all('tr'):
        cells = row.find_all(['th', 'td'])
        if len(cells) >= 2:
            label = cells[0].get_text(strip=True)
            value = cells[1].get_text(strip=True)
            print(f"  {label}: {value[:80]}")

In [ ]:
# Show the bio section
bio = extract_bio_section(text)
if bio:
    print("=== Bio Section (first 400 chars) ===")
    print(bio[:400])
else:
    print("No bio section found")

In [ ]:
# Show the sentencing section
sentence = extract_sentence_section(text)
if sentence:
    print("=== Sentencing Section (first 400 chars) ===")
    print(sentence[:400])
else:
    print("No sentencing section found")

## 2. Step-by-Step Extraction

Walk through each extraction stage on a single case.

In [ ]:
# Stage 1: Structured fields
data = extract_structured_fields(soup, case_id=100480)
print("Structured fields:")
print(f"  Court: {data.court}")
print(f"  Judge: {data.judge}")
print(f"  Date: {data.case_date}")
print(f"  Crime article: {data.crime_article}")
print(f"  Case number: {data.case_number}")
print(f"  Prosecutor: {data.prosecutor}")

In [ ]:
# Stage 2: Demographics (regex)
data.bio_text = bio
data.full_text = text
data = extract_demographics_regex(text, data)

print("Demographics:")
print(f"  Gender: {data.gender}")
print(f"  Age: {data.age}")
print(f"  Birth date: {data.birth_date}")
print(f"  Education: {data.education} (level {data.education_level})")
print(f"  Occupation: {data.occupation}")
print(f"  Employed: {data.employed} ({data.employment_detail})")
print(f"  Family size: {data.family_size}")
print(f"  Prior criminal: {data.prior_criminal}")

In [ ]:
# Stage 3: Sentencing (regex)
data.sentence_text = sentence
data = extract_sentence_regex(text, data)

print("Sentencing:")
print(f"  Type: {data.sentence_type}")
print(f"  Months: {data.sentence_months}")
print(f"  Fine (MNT): {data.sentence_fine_mnt}")
print(f"  Fine (units): {data.sentence_fine_units}")
print(f"  Community hours: {data.sentence_community_hours}")
print(f"  Suspended: {data.sentence_suspended}")

In [ ]:
# Full pipeline in one call
data_full = extract_case(html, case_id=100480, use_llm=False)
print(f"Full extraction: gender={data_full.gender}, age={data_full.age}, "
      f"edu={data_full.education}, sent={data_full.sentence_type}, "
      f"fine={data_full.sentence_fine_mnt}")

## 3. Batch Extraction

Run extraction on all 500 pilot cases.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

results = process_batch(
    input_dir=Path('../data/pilot'),
    output_path=Path('../data/pilot/extracted.json'),
    use_llm=False,
)
print(f"\nExtracted {len(results)} cases")

In [ ]:
# Load results into DataFrame for analysis
df = pd.DataFrame(results)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

## 4. Field Coverage

In [ ]:
# Count non-null values per field
key_fields = ['gender', 'age', 'education', 'occupation', 'employed',
              'family_size', 'prior_criminal', 'sentence_type',
              'sentence_months', 'sentence_fine_mnt']

coverage = pd.DataFrame({
    'non_null': df[key_fields].notna().sum(),
    'pct': (df[key_fields].notna().sum() / len(df) * 100).round(1),
})
print(coverage.to_string())

In [ ]:
# Distribution summaries
print("=== Gender ===")
print(df['gender'].value_counts(dropna=False))
print()

print("=== Sentence Type ===")
print(df['sentence_type'].value_counts(dropna=False))
print()

print("=== Education ===")
print(df['education'].value_counts(dropna=False))
print()

print("=== Age (descriptive) ===")
print(df['age'].describe())

## 5. Validation Results

We validate the automated extraction against an independent regex-based validator
(`scripts/validate_extraction.py`) on a random sample of 200 cases.

**Accuracy** = matches / (cases where at least one method found a value)

In [ ]:
# Load validation results
val_results = json.loads(Path('../data/pilot/validation_results.json').read_text())

print(f"Sample size: {val_results['metadata']['sample_size']}")
print(f"Total cases: {val_results['metadata']['total_cases']}")
print()

# Build accuracy table
rows = []
for field, stats in val_results['summary'].items():
    rows.append({
        'Field': field,
        'Match': stats['match'],
        'Mismatch': stats['mismatch'],
        'Auto-only': stats['auto_only'],
        'Manual-only': stats['manual_only'],
        'Both-null': stats['both_null'],
        'Accuracy': f"{stats['accuracy']:.1%}" if stats['accuracy'] else 'N/A',
    })

val_df = pd.DataFrame(rows)
print(val_df.to_string(index=False))

In [ ]:
# Show mismatches for investigation
if val_results['mismatches']:
    print(f"Mismatches ({len(val_results['mismatches'])} total):")
    for m in val_results['mismatches']:
        print(f"  case={m['case_id']}, field={m['field']}, "
              f"auto={m['auto']!r}, manual={m['manual']!r}")
else:
    print("No mismatches")

## 6. Gender Bug & Fix

**Bug:** The initial extraction searched for gender keywords in the full case text
with "эмэгтэй" (female) checked before "эрэгтэй" (male) due to dict ordering.
In multi-defendant cases, witness/victim descriptions containing "эмэгтэй" caused
male defendants to be misclassified as female (14 cases affected).

**Fix:** `_extract_gender_from_text()` isolates the first defendant's bio paragraph
(up to ~100 chars past the first "настай") and checks male before female.
Falls back to context-window search around demographic keywords.

In [ ]:
# Demonstrate the fix on case 130447 (was misclassified as female)
case_html = Path('../data/pilot/130447.html').read_text(encoding='utf-8')
case_data = extract_case(case_html, case_id=130447, use_llm=False)
print(f"Case 130447: gender={case_data.gender} (correctly male after fix)")